In [1]:
# ==========================================
# Project: Housing Dataset Analysis using AutoML Frameworks
# Author: BOUZID Mohamed El Khallil
# Specialty: Organic Process Engineering
# ==========================================

# 1. Install Libraries (تشغيل هذه الخلية أولاً)
!pip install h2o auto-sklearn tpot mljar-supervised scikit-learn pandas numpy

# ==========================================
# 2. Import & Setup
# ==========================================
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Load Dataset
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
results = {}

print("Data Loaded Successfully.")

# ==========================================
# 3. H2O AutoML
# ==========================================
import h2o
from h2o.automl import H2OAutoML

try:
    h2o.init()
    hf_train = h2o.H2Frame(pd.concat([X_train, y_train], axis=1))
    hf_test = h2o.H2Frame(pd.concat([X_test, y_test], axis=1))

    # Run H2O (Reduced time for demo)
    aml = H2OAutoML(max_models=5, seed=42, max_runtime_secs=120)
    aml.train(x=list(X.columns), y='target', training_frame=hf_train)

    preds_h2o = aml.predict(hf_test).as_data_frame()
    mse_h2o = mean_squared_error(y_test, preds_h2o)
    results['H2O'] = mse_h2o
    print(f"H2O MSE: {mse_h2o}")
except Exception as e:
    print(f"H2O Error: {e}")

# ==========================================
# 4. Auto-sklearn
# ==========================================
import autosklearn.regression

try:
    automl_sklearn = autosklearn.regression.AutoSklearnRegressor(
        time_left_for_this_task=120,
        per_run_time_limit=30
    )
    automl_sklearn.fit(X_train, y_train)
    y_pred_sklearn = automl_sklearn.predict(X_test)
    mse_sklearn = mean_squared_error(y_test, y_pred_sklearn)
    results['Auto-sklearn'] = mse_sklearn
    print(f"Auto-sklearn MSE: {mse_sklearn}")
except Exception as e:
    print(f"Auto-sklearn Error: {e}")

# ==========================================
# 5. TPOT
# ==========================================
from tpot import TPOTRegressor

try:
    tpot = TPOTRegressor(generations=2, population_size=10, verbosity=2, random_state=42)
    tpot.fit(X_train, y_train)
    y_pred_tpot = tpot.predict(X_test)
    mse_tpot = mean_squared_error(y_test, y_pred_tpot)
    results['TPOT'] = mse_tpot
    print(f"TPOT MSE: {mse_tpot}")
except Exception as e:
    print(f"TPOT Error: {e}")

# ==========================================
# 6. MLJAR
# ==========================================
from supervised.automl import AutoML

try:
    automl_mljar = AutoML(mode="Explain")
    automl_mljar.fit(X_train, y_train)
    y_pred_mljar = automl_mljar.predict(X_test)
    mse_mljar = mean_squared_error(y_test, y_pred_mljar)
    results['MLJAR'] = mse_mljar
    print(f"MLJAR MSE: {mse_mljar}")
except Exception as e:
    print(f"MLJAR Error: {e}")

# ==========================================
# 7. Final Results
# ==========================================
print("\n--- Final MSE Comparison (Lower is Better) ---")
df_results = pd.DataFrame(list(results.items()), columns=['Framework', 'MSE'])
print(df_results.sort_values(by='MSE'))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 37.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.3/127.3 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 96.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (pyproject.toml) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for deta

ModuleNotFoundError: No module named 'h2o'